In [2]:
import numpy as np
from astropy.io import fits
import glob
import matplotlib.pyplot as plt
import pandas as pd
import os

In [7]:
archivos=glob.glob('DATA_06MAST/DATA_????.fits', recursive=True)

In [4]:
#archivos

In [12]:
archivos=glob.glob('DATA_06MAST/DATA_????.fits', recursive=True)

tabla_datos = []

for f in archivos:
    header = fits.getheader(f)
    
    fila = {
        'Fichero': os.path.basename(f),
        'Objeto':  header.get('OBJECT', 'N/A'),
        'RA':      header.get('OBJCTRA', 'N/A'),
        'DEC':     header.get('OBJCTDEC', 'N/A'),
        'Exp':     header.get('EXPTIME', 0),
        'Fecha':   header.get('DATE-OBS', 'N/A'),
        'Airmass': header.get('AIRMASS', 'N/A'),
        'Tipo':    header.get('IMAGETYP', 'N/A')
    }
    
    tabla_datos.append(fila)

df = pd.DataFrame(tabla_datos)

df.to_csv('tabla de datos.csv', sep=';', index=False, encoding='utf-8-sig')

df.head()

,Fichero,Objeto,RA,DEC,Exp,Fecha,Airmass,Tipo
0,DATA_0001.fits,BIAS,13 41 34,+35 18 50,0.0,2021-07-04T17:11:09.00,1.0,Bias Frame
1,DATA_0002.fits,BIAS,N/A,N/A,0.0,2021-07-05T02:27:00.21,N/A,Bias Frame
2,DATA_0003.fits,BIAS,N/A,N/A,0.0,2021-07-05T02:27:38.15,N/A,Bias Frame
3,DATA_0004.fits,BIAS,N/A,N/A,0.0,2021-07-05T02:27:22.02,N/A,Bias Frame
4,DATA_0005.fits,BIAS,N/A,N/A,0.0,2021-07-05T02:27:16.65,N/A,Bias Frame


In [29]:
bias=[]
datos_bias=[]
flat=[]
datos_flat=[]
ciencia=[]
datos_ciencia=[]

for f in archivos:
    header=fits.getheader(f)
    datos=fits.getdata(f)
    objeto=header['OBJECT']
    if objeto == 'BIAS':
        bias.append(f)
        datos_bias.append(datos)
    elif objeto == 'FLAT':
        flat.append(f)
        datos_flat.append(datos)
    else:
        ciencia.append(f)
        datos_ciencia.append(datos)
        
df_bias = pd.DataFrame(bias)
df_flat = pd.DataFrame(flat)
df_ciencia = pd.DataFrame(ciencia)

df_bias.to_csv('lista de bias.txt', index=False, header=False)
df_flat.to_csv('lista de flat.txt', index=False, header=False)
df_ciencia.to_csv('lista de ciencia.txt', index=False, header=False)


In [20]:
bias_media=np.average(datos_bias, axis=0)
bias_mediana=np.median(datos_bias, axis=0)

new_bias_media = fits.PrimaryHDU(bias_media)
new_bias_mediana = fits.PrimaryHDU(bias_mediana)

new_bias_media.writeto('masterbias_media.fits')
new_bias_mediana.writeto('masterbias_mediana.fits')

**4.- Indica el número medio de cuentas del “master bias” y su desviación estándar. Escribe las funciones utilizadas para ello.**

In [26]:
#número de cuentas y desviación estándar usando la media:
imagen_masterbias_media = fits.open('masterbias_media.fits')
datos_masterbias_media = imagen_masterbias_media[0].data

media = np.mean(datos_masterbias_media)
desviacion_media = np.std(datos_masterbias_media)

print(f"  - Nivel de cuentas (Media):   {media:.4f} ADUs")
print(f"  - Desviación estándar (RMS):   {desviacion_media:.4f} ADUs")

  - Nivel de cuentas (Media):   301.2248 ADUs
  - Desviación estándar (RMS):   1.4106 ADUs


#número de cuentas y desviación estándar usando la mediana:
imagen_masterbias_mediana = fits.open('masterbias_mediana.fits')
datos_masterbias_mediana = imagen_masterbias_mediana[0].data

mediana = np.mean(datos_masterbias_mediana)
desviacion_mediana = np.std(datos_masterbias_mediana)

print(f"  - Nivel de cuentas (Mediana):   {mediana:.4f} ADUs")
print(f"  - Desviación estándar (RMS):   {desviacion_mediana:.4f} ADUs")

**5.- Corrige de bias todas las imágenes. Escribe las funciones utilizadas para ello.**


Corrección de flats de bias (usando la media):

In [30]:
for elemento in flat:
    datos=fits.getdata(elemento)
    header=fits.getheader(elemento)
    datos_bias=datos-bias_media
    header['HISTORY']='corregido de bias con la media'
    newflat=fits.PrimaryHDU(datos_bias,header=header)
    newflat.writeto(elemento[:-5]+'_media_bias.fits')

Corrección de flats de bias (usando la mediana):

In [31]:
for elemento in flat:
    datos=fits.getdata(elemento)
    header=fits.getheader(elemento)
    datos_bias=datos-bias_mediana
    header['HISTORY']='corregido de bias con la mediana'
    newflatmediana=fits.PrimaryHDU(datos_bias,header=header)
    newflatmediana.writeto(elemento[:-5]+'_mediana_bias.fits')

Corrección a datos de ciencia de bias (usando la media):

In [33]:
for elemento in ciencia:
    datos=fits.getdata(elemento)
    header=fits.getheader(elemento)
    datos_bias=datos-bias_media
    header['HISTORY']='corregido de bias con la media'
    newciencia=fits.PrimaryHDU(datos_bias,header=header)
    newciencia.writeto(elemento[:-5]+'_media_bias.fits')

Corrección a datos de ciencia de bias (usando la mediana):

In [37]:
for elemento in ciencia:
    datos=fits.getdata(elemento)
    header=fits.getheader(elemento)
    datos_bias=datos-bias_mediana
    header['HISTORY']='corregido de bias con la mediana'
    newcienciamediana=fits.PrimaryHDU(datos_bias,header=header)
    newcienciamediana.writeto(elemento[:-5]+'_mediana_bias.fits', overwrite=True)

**6.- Obtén el “master flat” para los filtros B, V, R e I utilizando la media y la mediana, 
escribe las funciones utilizadas para ello.** 

In [43]:
#lista que contiene las imágenes correginas de bias (con la media)
lista_media_bias = glob.glob('DATA_06MAST/DATA*media_bias.fits')

#lista que contiene las imágenes correginas de bias (con la mediana)
lista_mediana_bias = glob.glob('DATA_06MAST/DATA*mediana_bias.fits')

In [47]:
flat_media_B = []
flat_media_V = []
flat_media_R = []
flat_media_I = []
ciencia_media_B = []
ciencia_media_V = []
ciencia_media_R = []
ciencia_media_I = []

flat_mediana_B = []
flat_mediana_V = []
flat_mediana_R = []
flat_mediana_I = []
ciencia_mediana_B = []
ciencia_mediana_V = []
ciencia_mediana_R = []
ciencia_mediana_I = []

for elemento in lista_media_bias:
    header=fits.getheader(elemento)
    filtro=header['FILTER']
    objeto=header['OBJECT']
    if objeto == 'FLAT':
        if filtro == 'B':
            flat_media_B.append(elemento)
        elif filtro == 'V':
            flat_media_V.append(elemento)
        elif filtro == 'R':
            flat_media_R.append(elemento)
        else: 
            flat_media_I.append(elemento)
    else:
        if filtro == 'B':
            ciencia_media_B.append(elemento)
        elif filtro == 'V':
            ciencia_media_V.append(elemento)
        elif filtro == 'R':
            ciencia_media_R.append(elemento)
        else: 
            ciencia_media_I.append(elemento)

for elemento in lista_mediana_bias:
    header=fits.getheader(elemento)
    filtro=header['FILTER']
    objeto=header['OBJECT']
    if objeto == 'FLAT':
        if filtro == 'B':
            flat_mediana_B.append(elemento)
        elif filtro == 'V':
            flat_mediana_V.append(elemento)
        elif filtro == 'R':
            flat_mediana_R.append(elemento)
        else: 
            flat_mediana_I.append(elemento)
    else:
        if filtro == 'B':
            ciencia_mediana_B.append(elemento)
        elif filtro == 'V':
            ciencia_mediana_V.append(elemento)
        elif filtro == 'R':
            ciencia_mediana_R.append(elemento)
        else: 
            ciencia_mediana_I.append(elemento)

In [49]:
flat_media_V

['DATA_06MAST\\DATA_0021_media_bias.fits',
 'DATA_06MAST\\DATA_0022_media_bias.fits',
 'DATA_06MAST\\DATA_0023_media_bias.fits',
 'DATA_06MAST\\DATA_0024_media_bias.fits',
 'DATA_06MAST\\DATA_0025_media_bias.fits',
 'DATA_06MAST\\DATA_0026_media_bias.fits',
 'DATA_06MAST\\DATA_0027_media_bias.fits']

In [50]:
flat_mediana_V

['DATA_06MAST\\DATA_0021_mediana_bias.fits',
 'DATA_06MAST\\DATA_0022_mediana_bias.fits',
 'DATA_06MAST\\DATA_0023_mediana_bias.fits',
 'DATA_06MAST\\DATA_0024_mediana_bias.fits',
 'DATA_06MAST\\DATA_0025_mediana_bias.fits',
 'DATA_06MAST\\DATA_0026_mediana_bias.fits',
 'DATA_06MAST\\DATA_0027_mediana_bias.fits']

In [51]:
flat_media_B_data = []
flat_media_V_data = []
flat_media_R_data = []
flat_media_I_data = []

flat_mediana_B_data = []
flat_mediana_V_data = []
flat_mediana_R_data = []
flat_mediana_I_data = []

for elemento in flat_media_B:
    imagen = fits.getdata(elemento)
    flat_media_B_data.append(imagen)

for elemento in flat_media_V:
    imagen = fits.getdata(elemento)
    flat_media_V_data.append(imagen)

for elemento in flat_media_R:
    imagen = fits.getdata(elemento)
    flat_media_R_data.append(imagen)

for elemento in flat_media_I:
    imagen = fits.getdata(elemento)
    flat_media_I_data.append(imagen)

for elemento in flat_mediana_B:
    imagen = fits.getdata(elemento)
    flat_mediana_B_data.append(imagen)

for elemento in flat_mediana_V:
    imagen = fits.getdata(elemento)
    flat_mediana_V_data.append(imagen)

for elemento in flat_mediana_R:
    imagen = fits.getdata(elemento)
    flat_mediana_R_data.append(imagen)

for elemento in flat_mediana_I:
    imagen = fits.getdata(elemento)
    flat_mediana_I_data.append(imagen)

In [55]:
#flats medios usando la media (con los bias usando la media):
flat_media_B_media = np.average(flat_media_B_data, axis=0)
flat_media_V_media = np.average(flat_media_V_data, axis=0)
flat_media_R_media = np.average(flat_media_R_data, axis=0)
flat_media_I_media = np.average(flat_media_I_data, axis=0)

#flats medios usando la media (con los bias usando la mediana):
flat_mediana_B_media = np.average(flat_mediana_B_data, axis=0)
flat_mediana_V_media = np.average(flat_mediana_V_data, axis=0)
flat_mediana_R_media = np.average(flat_mediana_R_data, axis=0)
flat_mediana_I_media = np.average(flat_mediana_I_data, axis=0)

#flats medios usando la mediana (con los bias usando la media):
flat_media_B_mediana = np.median(flat_media_B_data, axis=0)
flat_media_V_mediana = np.median(flat_media_V_data, axis=0)
flat_media_R_mediana = np.median(flat_media_R_data, axis=0)
flat_media_I_mediana = np.median(flat_media_I_data, axis=0)

#flats medios usando la mediana (con los bias usando la mediana):
flat_mediana_B_mediana = np.median(flat_mediana_B_data, axis=0)
flat_mediana_V_mediana = np.median(flat_mediana_V_data, axis=0)
flat_mediana_R_mediana = np.median(flat_mediana_R_data, axis=0)
flat_mediana_I_mediana = np.median(flat_mediana_I_data, axis=0)

**7.- Normaliza el “master flat” para los filtros B, V, R e I utilizando la media y la 
mediana, escribe las funciones utilizadas para ello.**

In [56]:
#flats normalizados:

flat_media_B_media_n = flat_media_B_media/np.average(flat_media_B_media, axis=0)
flat_media_V_media_n = flat_media_V_media/np.average(flat_media_V_media, axis=0)
flat_media_R_media_n = flat_media_R_media/np.average(flat_media_R_media, axis=0)
flat_media_I_media_n = flat_media_I_media/np.average(flat_media_I_media, axis=0)

flat_mediana_B_media_n = flat_mediana_B_media/np.average(flat_mediana_B_media, axis=0)
flat_mediana_V_media_n = flat_mediana_V_media/np.average(flat_mediana_V_media, axis=0)
flat_mediana_R_media_n = flat_mediana_R_media/np.average(flat_mediana_R_media, axis=0)
flat_mediana_I_media_n = flat_mediana_I_media/np.average(flat_mediana_I_media, axis=0)

flat_media_B_mediana_n = flat_media_B_mediana/np.average(flat_media_B_mediana, axis=0)
flat_media_V_mediana_n = flat_media_V_mediana/np.average(flat_media_V_mediana, axis=0)
flat_media_R_mediana_n = flat_media_R_mediana/np.average(flat_media_R_mediana, axis=0)
flat_media_I_mediana_n = flat_media_I_mediana/np.average(flat_media_I_mediana, axis=0)

flat_mediana_B_mediana_n = flat_mediana_B_mediana/np.average(flat_mediana_B_mediana, axis=0)
flat_mediana_V_mediana_n = flat_mediana_V_mediana/np.average(flat_mediana_V_mediana, axis=0)
flat_mediana_R_mediana_n = flat_mediana_R_mediana/np.average(flat_mediana_R_mediana, axis=0)
flat_mediana_I_mediana_n = flat_mediana_I_mediana/np.average(flat_mediana_I_mediana, axis=0)

#para comprobar:
print(f"El flat B normalizado es: {np.average(flat_media_B_media_n)}")
print(f"El flat V normalizado es: {np.average(flat_media_V_media_n)}")
print(f"El flat R normalizado es: {np.average(flat_media_R_media_n)}")
print(f"El flat I normalizado es: {np.average(flat_media_I_media_n)}")
print("-" *20)
print(f"El flat B normalizado es: {np.average(flat_mediana_B_media_n)}")
print(f"El flat V normalizado es: {np.average(flat_mediana_V_media_n)}")
print(f"El flat R normalizado es: {np.average(flat_mediana_R_media_n)}")
print(f"El flat I normalizado es: {np.average(flat_mediana_I_media_n)}")
print("-" *20)
print(f"El flat B normalizado es: {np.average(flat_media_B_mediana_n)}")
print(f"El flat V normalizado es: {np.average(flat_media_V_mediana_n)}")
print(f"El flat R normalizado es: {np.average(flat_media_R_mediana_n)}")
print(f"El flat I normalizado es: {np.average(flat_media_I_mediana_n)}")
print("-" *20)
print(f"El flat B normalizado es: {np.average(flat_mediana_B_mediana_n)}")
print(f"El flat V normalizado es: {np.average(flat_mediana_V_mediana_n)}")
print(f"El flat R normalizado es: {np.average(flat_mediana_R_mediana_n)}")
print(f"El flat I normalizado es: {np.average(flat_mediana_I_mediana_n)}")

El flat B normalizado es: 1.0000000000000007
El flat V normalizado es: 1.0000000000000004
El flat R normalizado es: 0.9999999999999996
El flat I normalizado es: 1.0
--------------------
El flat B normalizado es: 1.0
El flat V normalizado es: 1.000000000000001
El flat R normalizado es: 0.9999999999999998
El flat I normalizado es: 0.9999999999999996
--------------------
El flat B normalizado es: 1.0000000000000002
El flat V normalizado es: 0.9999999999999996
El flat R normalizado es: 1.0
El flat I normalizado es: 1.0000000000000007
--------------------
El flat B normalizado es: 1.0
El flat V normalizado es: 0.9999999999999998
El flat R normalizado es: 0.9999999999999996
El flat I normalizado es: 0.9999999999999996


In [57]:
'''
puede que haya un valor en el flat medio que sea igual a cero, 
para evitar indeterminaciones hay que cambiar estos ceros por unos:
'''
flat_media_B_media_n[flat_media_B_media_n == 0] = 1
flat_media_V_media_n[flat_media_V_media_n == 0] = 1
flat_media_R_media_n[flat_media_R_media_n == 0] = 1
flat_media_I_media_n[flat_media_I_media_n == 0] = 1

flat_mediana_B_media_n[flat_mediana_B_media_n == 0] = 1
flat_mediana_V_media_n[flat_mediana_V_media_n == 0] = 1
flat_mediana_R_media_n[flat_mediana_R_media_n == 0] = 1
flat_mediana_I_media_n[flat_mediana_I_media_n == 0] = 1

flat_media_B_mediana_n[flat_media_B_mediana_n == 0] = 1
flat_media_V_mediana_n[flat_media_V_mediana_n == 0] = 1
flat_media_R_mediana_n[flat_media_R_mediana_n == 0] = 1
flat_media_I_mediana_n[flat_media_I_mediana_n == 0] = 1

flat_mediana_B_mediana_n[flat_mediana_B_mediana_n == 0] = 1
flat_mediana_V_mediana_n[flat_mediana_V_mediana_n == 0] = 1
flat_mediana_R_mediana_n[flat_mediana_R_mediana_n == 0] = 1
flat_mediana_I_mediana_n[flat_mediana_I_mediana_n == 0] = 1


In [59]:
#creación de archivos fits con los flats normalizados:

f1nuevo = fits.PrimaryHDU(flat_media_B_media_n)
f1nuevo.writeto('flat_media_B_media_normalizado.fits')

f2nuevo = fits.PrimaryHDU(flat_media_V_media_n)
f2nuevo.writeto('flat_media_V_media_normalizado.fits')

f3nuevo = fits.PrimaryHDU(flat_media_R_media_n)
f3nuevo.writeto('flat_media_R_media_normalizado.fits')

f4nuevo = fits.PrimaryHDU(flat_media_I_media_n)
f4nuevo.writeto('flat_media_I_media_normalizado.fits')

f5nuevo = fits.PrimaryHDU(flat_mediana_B_media_n)
f5nuevo.writeto('flat_mediana_B_media_normalizado.fits')

f6nuevo = fits.PrimaryHDU(flat_mediana_V_media_n)
f6nuevo.writeto('flat_mediana_V_media_normalizado.fits')

f7nuevo = fits.PrimaryHDU(flat_mediana_R_media_n)
f7nuevo.writeto('flat_mediana_R_media_normalizado.fits')

f8nuevo = fits.PrimaryHDU(flat_mediana_I_media_n)
f8nuevo.writeto('flat_mediana_I_media_normalizado.fits')

f9nuevo = fits.PrimaryHDU(flat_media_B_mediana_n)
f9nuevo.writeto('flat_media_B_mediana_normalizado.fits')

f10nuevo = fits.PrimaryHDU(flat_media_V_mediana_n)
f10nuevo.writeto('flat_media_V_mediana_normalizado.fits')

f11nuevo = fits.PrimaryHDU(flat_media_R_mediana_n)
f11nuevo.writeto('flat_media_R_mediana_normalizado.fits')

f12nuevo = fits.PrimaryHDU(flat_media_I_mediana_n)
f12nuevo.writeto('flat_media_I_mediana_normalizado.fits')

f13nuevo = fits.PrimaryHDU(flat_mediana_B_mediana_n)
f13nuevo.writeto('flat_mediana_B_mediana_normalizado.fits')

f14nuevo = fits.PrimaryHDU(flat_mediana_V_mediana_n)
f14nuevo.writeto('flat_mediana_V_mediana_normalizado.fits')

f15nuevo = fits.PrimaryHDU(flat_mediana_R_mediana_n)
f15nuevo.writeto('flat_mediana_R_mediana_normalizado.fits')

f16nuevo = fits.PrimaryHDU(flat_mediana_I_mediana_n)
f16nuevo.writeto('flat_mediana_I_mediana_normalizado.fits')

In [3]:
!pip install ipympl
!pip install mplcursors

In [18]:
#visualización de un flat

%matplotlib qt
from astropy.visualization import ZScaleInterval, ImageNormalize
interval = ZScaleInterval()
vmin, vmax = interval.get_limits(flat_media_B_media_n)
plt.imshow(flat_media_B_media_n, cmap = 'gray', origin = 'lower', vmin = vmin, vmax = vmax)
plt.show()

NameError: name 'flat_media_B_media_n' is not defined

In [4]:
#función para visualizar los flat normalizados:

def visualizar_flat(imagen_flat_normalizado):
    %matplotlib qt
    from astropy.visualization import ZScaleInterval, ImageNormalize
    interval = ZScaleInterval()
    vmin, vmax = interval.get_limits(imagen_flat_normalizado)
    plt.imshow(imagen_flat_normalizado, cmap = 'Greys', origin = 'lower', 
               vmin = vmin, vmax = vmax)
    return plt.show()

In [105]:
#llamado de la función:

visualizar_flat(flat_mediana_V_mediana_n)

**8.- Corrige de flat todas las imágenes que así lo requieran, indica cuales son y las 
funciones utilizadas para ello.**

In [74]:
ciencia_media_B

['DATA_06MAST\\DATA_0051_media_bias.fits',
 'DATA_06MAST\\DATA_0052_media_bias.fits',
 'DATA_06MAST\\DATA_0056_media_bias.fits',
 'DATA_06MAST\\DATA_0057_media_bias.fits',
 'DATA_06MAST\\DATA_0060_media_bias.fits',
 'DATA_06MAST\\DATA_0065_media_bias.fits',
 'DATA_06MAST\\DATA_0068_media_bias.fits',
 'DATA_06MAST\\DATA_0086_media_bias.fits',
 'DATA_06MAST\\DATA_0089_media_bias.fits',
 'DATA_06MAST\\DATA_0095_media_bias.fits',
 'DATA_06MAST\\DATA_0097_media_bias.fits',
 'DATA_06MAST\\DATA_0098_media_bias.fits',
 'DATA_06MAST\\DATA_0107_media_bias.fits',
 'DATA_06MAST\\DATA_0110_media_bias.fits',
 'DATA_06MAST\\DATA_0111_media_bias.fits',
 'DATA_06MAST\\DATA_0113_media_bias.fits',
 'DATA_06MAST\\DATA_0119_media_bias.fits',
 'DATA_06MAST\\DATA_0120_media_bias.fits',
 'DATA_06MAST\\DATA_0121_media_bias.fits',
 'DATA_06MAST\\DATA_0122_media_bias.fits',
 'DATA_06MAST\\DATA_0133_media_bias.fits',
 'DATA_06MAST\\DATA_0136_media_bias.fits',
 'DATA_06MAST\\DATA_0137_media_bias.fits',
 'DATA_06MA


Las imágenes que necesitan correción de flats son las imágenes de ciencia que previamente
fueron corregidas de bias (tanto con la media, como con la mediana).


In [86]:
#corrección de flats para el filtro B:
for elemento in ciencia_media_B:
    data = fits.getdata(elemento)
    header = fits.getheader(elemento)
    header['HISTORY2'] = 'corregido de flats'
    #imagen de ciencia corregida de bias media, y flat (corregido de bias media) media:
    ciencia_media_B_flat_media_B_media_n = data/flat_media_B_media_n
    newsciflat = fits.PrimaryHDU(ciencia_media_B_flat_media_B_media_n, header=header)
    newsciflat.writeto(elemento[:-5]+'_media_media_flat.fits')
    #imagen de ciencia corregida de bias media, y flat (corregido de bias mediana) media:
    ciencia_media_B_flat_mediana_B_media_n = data/flat_mediana_B_media_n
    newsciflat2 = fits.PrimaryHDU(ciencia_media_B_flat_mediana_B_media_n, header=header)
    newsciflat2.writeto(elemento[:-5]+'_mediana_media_flat.fits')
    #imagen de ciencia corregida de bias media, y flat (corregido de bias media) mediana:
    ciencia_media_B_flat_media_B_mediana_n = data/flat_media_B_mediana_n
    newsciflat3 = fits.PrimaryHDU(ciencia_media_B_flat_media_B_mediana_n, header=header)
    newsciflat3.writeto(elemento[:-5]+'_media_mediana_flat.fits')
    #imagen de ciencia corregida de bias media, y flat (corregido de bias mediana) mediana:
    ciencia_media_B_flat_mediana_B_mediana_n = data/flat_mediana_B_mediana_n
    newsciflat4 = fits.PrimaryHDU(ciencia_media_B_flat_mediana_B_mediana_n, header=header)
    newsciflat4.writeto(elemento[:-5]+'_mediana_mediana_flat.fits')

In [87]:
#corrección de flats para el filtro V:
for elemento in ciencia_media_V:
    data = fits.getdata(elemento)
    header = fits.getheader(elemento)
    header['HISTORY2'] = 'corregido de flats'
    #imagen de ciencia corregida de bias media, y flat (corregido de bias media) media:
    ciencia_media_V_flat_media_V_media_n = data/flat_media_V_media_n
    newsciflat = fits.PrimaryHDU(ciencia_media_V_flat_media_V_media_n, header=header)
    newsciflat.writeto(elemento[:-5]+'_media_media_flat.fits')
    #imagen de ciencia corregida de bias media, y flat (corregido de bias mediana) media:
    ciencia_media_V_flat_mediana_V_media_n = data/flat_mediana_V_media_n
    newsciflat2 = fits.PrimaryHDU(ciencia_media_V_flat_mediana_V_media_n, header=header)
    newsciflat2.writeto(elemento[:-5]+'_mediana_media_flat.fits')
    #imagen de ciencia corregida de bias media, y flat (corregido de bias media) mediana:
    ciencia_media_V_flat_media_V_mediana_n = data/flat_media_V_mediana_n
    newsciflat3 = fits.PrimaryHDU(ciencia_media_V_flat_media_V_mediana_n, header=header)
    newsciflat3.writeto(elemento[:-5]+'_media_mediana_flat.fits')
    #imagen de ciencia corregida de bias media, y flat (corregido de bias mediana) mediana:
    ciencia_media_V_flat_mediana_V_mediana_n = data/flat_mediana_V_mediana_n
    newsciflat4 = fits.PrimaryHDU(ciencia_media_V_flat_mediana_V_mediana_n, header=header)
    newsciflat4.writeto(elemento[:-5]+'_mediana_mediana_flat.fits')

In [88]:
#corrección de flats para el filtro R:
for elemento in ciencia_media_R:
    data = fits.getdata(elemento)
    header = fits.getheader(elemento)
    header['HISTORY2'] = 'corregido de flats'
    #imagen de ciencia corregida de bias media, y flat (corregido de bias media) media:
    ciencia_media_R_flat_media_R_media_n = data/flat_media_R_media_n
    newsciflat = fits.PrimaryHDU(ciencia_media_R_flat_media_R_media_n, header=header)
    newsciflat.writeto(elemento[:-5]+'_media_media_flat.fits')
    #imagen de ciencia corregida de bias media, y flat (corregido de bias mediana) media:
    ciencia_media_R_flat_mediana_R_media_n = data/flat_mediana_R_media_n
    newsciflat2 = fits.PrimaryHDU(ciencia_media_R_flat_mediana_R_media_n, header=header)
    newsciflat2.writeto(elemento[:-5]+'_mediana_media_flat.fits')
    #imagen de ciencia corregida de bias media, y flat (corregido de bias media) mediana:
    ciencia_media_R_flat_media_R_mediana_n = data/flat_media_R_mediana_n
    newsciflat3 = fits.PrimaryHDU(ciencia_media_R_flat_media_R_mediana_n, header=header)
    newsciflat3.writeto(elemento[:-5]+'_media_mediana_flat.fits')
    #imagen de ciencia corregida de bias media, y flat (corregido de bias mediana) mediana:
    ciencia_media_R_flat_mediana_R_mediana_n = data/flat_mediana_R_mediana_n
    newsciflat4 = fits.PrimaryHDU(ciencia_media_R_flat_mediana_R_mediana_n, header=header)
    newsciflat4.writeto(elemento[:-5]+'_mediana_mediana_flat.fits')

In [89]:
#corrección de flats para el filtro I:
for elemento in ciencia_media_I:
    data = fits.getdata(elemento)
    header = fits.getheader(elemento)
    header['HISTORY2'] = 'corregido de flats'
    #imagen de ciencia corregida de bias media, y flat (corregido de bias media) media:
    ciencia_media_I_flat_media_I_media_n = data/flat_media_I_media_n
    newsciflat = fits.PrimaryHDU(ciencia_media_I_flat_media_I_media_n, header=header)
    newsciflat.writeto(elemento[:-5]+'_media_media_flat.fits')
    #imagen de ciencia corregida de bias media, y flat (corregido de bias mediana) media:
    ciencia_media_I_flat_mediana_I_media_n = data/flat_mediana_I_media_n
    newsciflat2 = fits.PrimaryHDU(ciencia_media_I_flat_mediana_I_media_n, header=header)
    newsciflat2.writeto(elemento[:-5]+'_mediana_media_flat.fits')
    #imagen de ciencia corregida de bias media, y flat (corregido de bias media) mediana:
    ciencia_media_I_flat_media_I_mediana_n = data/flat_media_I_mediana_n
    newsciflat3 = fits.PrimaryHDU(ciencia_media_I_flat_media_I_mediana_n, header=header)
    newsciflat3.writeto(elemento[:-5]+'_media_mediana_flat.fits')
    #imagen de ciencia corregida de bias media, y flat (corregido de bias mediana) mediana:
    ciencia_media_I_flat_mediana_I_mediana_n = data/flat_mediana_I_mediana_n
    newsciflat4 = fits.PrimaryHDU(ciencia_media_I_flat_mediana_I_mediana_n, header=header)
    newsciflat4.writeto(elemento[:-5]+'_mediana_mediana_flat.fits')

IMÁGENES DE CIENCIA QUE FUERON CORREGIDAS DE BIAS CON LA MEDIANA:

In [90]:
#corrección de flats para el filtro B:
for elemento in ciencia_mediana_B:
    data = fits.getdata(elemento)
    header = fits.getheader(elemento)
    header['HISTORY2'] = 'corregido de flats'
    #imagen de ciencia corregida de bias media, y flat (corregido de bias media) media:
    ciencia_mediana_B_flat_media_B_media_n = data/flat_media_B_media_n
    newsciflat = fits.PrimaryHDU(ciencia_mediana_B_flat_media_B_media_n, header=header)
    newsciflat.writeto(elemento[:-5]+'_media_media_flat.fits')
    #imagen de ciencia corregida de bias media, y flat (corregido de bias mediana) media:
    ciencia_mediana_B_flat_mediana_B_media_n = data/flat_mediana_B_media_n
    newsciflat2 = fits.PrimaryHDU(ciencia_mediana_B_flat_mediana_B_media_n, header=header)
    newsciflat2.writeto(elemento[:-5]+'_mediana_media_flat.fits')
    #imagen de ciencia corregida de bias media, y flat (corregido de bias media) mediana:
    ciencia_mediana_B_flat_media_B_mediana_n = data/flat_media_B_mediana_n
    newsciflat3 = fits.PrimaryHDU(ciencia_mediana_B_flat_media_B_mediana_n, header=header)
    newsciflat3.writeto(elemento[:-5]+'_media_mediana_flat.fits')
    #imagen de ciencia corregida de bias media, y flat (corregido de bias mediana) mediana:
    ciencia_mediana_B_flat_mediana_B_mediana_n = data/flat_mediana_B_mediana_n
    newsciflat4 = fits.PrimaryHDU(ciencia_mediana_B_flat_mediana_B_mediana_n, header=header)
    newsciflat4.writeto(elemento[:-5]+'_mediana_mediana_flat.fits')

In [91]:
#corrección de flats para el filtro V:
for elemento in ciencia_mediana_V:
    data = fits.getdata(elemento)
    header = fits.getheader(elemento)
    header['HISTORY2'] = 'corregido de flats'
    #imagen de ciencia corregida de bias media, y flat (corregido de bias media) media:
    ciencia_mediana_V_flat_media_V_media_n = data/flat_media_V_media_n
    newsciflat = fits.PrimaryHDU(ciencia_mediana_V_flat_media_V_media_n, header=header)
    newsciflat.writeto(elemento[:-5]+'_media_media_flat.fits')
    #imagen de ciencia corregida de bias media, y flat (corregido de bias mediana) media:
    ciencia_mediana_V_flat_mediana_V_media_n = data/flat_mediana_V_media_n
    newsciflat2 = fits.PrimaryHDU(ciencia_mediana_V_flat_mediana_V_media_n, header=header)
    newsciflat2.writeto(elemento[:-5]+'_mediana_media_flat.fits')
    #imagen de ciencia corregida de bias media, y flat (corregido de bias media) mediana:
    ciencia_mediana_V_flat_media_V_mediana_n = data/flat_media_V_mediana_n
    newsciflat3 = fits.PrimaryHDU(ciencia_mediana_V_flat_media_V_mediana_n, header=header)
    newsciflat3.writeto(elemento[:-5]+'_media_mediana_flat.fits')
    #imagen de ciencia corregida de bias media, y flat (corregido de bias mediana) mediana:
    ciencia_mediana_V_flat_mediana_V_mediana_n = data/flat_mediana_V_mediana_n
    newsciflat4 = fits.PrimaryHDU(ciencia_mediana_V_flat_mediana_V_mediana_n, header=header)
    newsciflat4.writeto(elemento[:-5]+'_mediana_mediana_flat.fits')

In [92]:
#corrección de flats para el filtro R:
for elemento in ciencia_mediana_R:
    data = fits.getdata(elemento)
    header = fits.getheader(elemento)
    header['HISTORY2'] = 'corregido de flats'
    #imagen de ciencia corregida de bias media, y flat (corregido de bias media) media:
    ciencia_mediana_R_flat_media_R_media_n = data/flat_media_R_media_n
    newsciflat = fits.PrimaryHDU(ciencia_mediana_R_flat_media_R_media_n, header=header)
    newsciflat.writeto(elemento[:-5]+'_media_media_flat.fits')
    #imagen de ciencia corregida de bias media, y flat (corregido de bias mediana) media:
    ciencia_mediana_R_flat_mediana_R_media_n = data/flat_mediana_R_media_n
    newsciflat2 = fits.PrimaryHDU(ciencia_mediana_R_flat_mediana_R_media_n, header=header)
    newsciflat2.writeto(elemento[:-5]+'_mediana_media_flat.fits')
    #imagen de ciencia corregida de bias media, y flat (corregido de bias media) mediana:
    ciencia_mediana_R_flat_media_R_mediana_n = data/flat_media_R_mediana_n
    newsciflat3 = fits.PrimaryHDU(ciencia_mediana_R_flat_media_R_mediana_n, header=header)
    newsciflat3.writeto(elemento[:-5]+'_media_mediana_flat.fits')
    #imagen de ciencia corregida de bias media, y flat (corregido de bias mediana) mediana:
    ciencia_mediana_R_flat_mediana_R_mediana_n = data/flat_mediana_R_mediana_n
    newsciflat4 = fits.PrimaryHDU(ciencia_mediana_R_flat_mediana_R_mediana_n, header=header)
    newsciflat4.writeto(elemento[:-5]+'_mediana_mediana_flat.fits')

In [93]:
#corrección de flats para el filtro I:
for elemento in ciencia_mediana_I:
    data = fits.getdata(elemento)
    header = fits.getheader(elemento)
    header['HISTORY2'] = 'corregido de flats'
    #imagen de ciencia corregida de bias media, y flat (corregido de bias media) media:
    ciencia_mediana_I_flat_media_I_media_n = data/flat_media_I_media_n
    newsciflat = fits.PrimaryHDU(ciencia_mediana_I_flat_media_I_media_n, header=header)
    newsciflat.writeto(elemento[:-5]+'_media_media_flat.fits')
    #imagen de ciencia corregida de bias media, y flat (corregido de bias mediana) media:
    ciencia_mediana_I_flat_mediana_I_media_n = data/flat_mediana_I_media_n
    newsciflat2 = fits.PrimaryHDU(ciencia_mediana_I_flat_mediana_I_media_n, header=header)
    newsciflat2.writeto(elemento[:-5]+'_mediana_media_flat.fits')
    #imagen de ciencia corregida de bias media, y flat (corregido de bias media) mediana:
    ciencia_mediana_I_flat_media_I_mediana_n = data/flat_media_I_mediana_n
    newsciflat3 = fits.PrimaryHDU(ciencia_mediana_I_flat_media_I_mediana_n, header=header)
    newsciflat3.writeto(elemento[:-5]+'_media_mediana_flat.fits')
    #imagen de ciencia corregida de bias media, y flat (corregido de bias mediana) mediana:
    ciencia_mediana_I_flat_mediana_I_mediana_n = data/flat_mediana_I_mediana_n
    newsciflat4 = fits.PrimaryHDU(ciencia_mediana_I_flat_mediana_I_mediana_n, header=header)
    newsciflat4.writeto(elemento[:-5]+'_mediana_mediana_flat.fits')

In [108]:
#funcion para visualizar una imagen de ciencia .fits ya coregido de flat:
#misma función que antes, solo que ahora se abre primero el fichero .fits y 
#se guarda en una variable:

imagen_sci = fits.getdata('DATA_06MAST/DATA_0088_mediana_bias_mediana_mediana_flat.fits')
#llamado a la función:
visualizar_flat(imagen_sci)

**9.- Elige una imagen de ciencia para cada filtro y obtén un perfil de 5 de las estrellas 
presentes en la imagen, haz una tabla con la siguiente información:**

In [113]:
#visualizar las listas de ciencia_mediana para saber cuales imágenes son de cada filtro:

In [111]:
ciencia_mediana_B

['DATA_06MAST\\DATA_0051_mediana_bias.fits',
 'DATA_06MAST\\DATA_0052_mediana_bias.fits',
 'DATA_06MAST\\DATA_0056_mediana_bias.fits',
 'DATA_06MAST\\DATA_0057_mediana_bias.fits',
 'DATA_06MAST\\DATA_0060_mediana_bias.fits',
 'DATA_06MAST\\DATA_0065_mediana_bias.fits',
 'DATA_06MAST\\DATA_0068_mediana_bias.fits',
 'DATA_06MAST\\DATA_0086_mediana_bias.fits',
 'DATA_06MAST\\DATA_0089_mediana_bias.fits',
 'DATA_06MAST\\DATA_0095_mediana_bias.fits',
 'DATA_06MAST\\DATA_0097_mediana_bias.fits',
 'DATA_06MAST\\DATA_0098_mediana_bias.fits',
 'DATA_06MAST\\DATA_0107_mediana_bias.fits',
 'DATA_06MAST\\DATA_0110_mediana_bias.fits',
 'DATA_06MAST\\DATA_0111_mediana_bias.fits',
 'DATA_06MAST\\DATA_0113_mediana_bias.fits',
 'DATA_06MAST\\DATA_0119_mediana_bias.fits',
 'DATA_06MAST\\DATA_0120_mediana_bias.fits',
 'DATA_06MAST\\DATA_0121_mediana_bias.fits',
 'DATA_06MAST\\DATA_0122_mediana_bias.fits',
 'DATA_06MAST\\DATA_0133_mediana_bias.fits',
 'DATA_06MAST\\DATA_0136_mediana_bias.fits',
 'DATA_06M

In [114]:
ciencia_mediana_V

['DATA_06MAST\\DATA_0053_mediana_bias.fits',
 'DATA_06MAST\\DATA_0054_mediana_bias.fits',
 'DATA_06MAST\\DATA_0055_mediana_bias.fits',
 'DATA_06MAST\\DATA_0058_mediana_bias.fits',
 'DATA_06MAST\\DATA_0063_mediana_bias.fits',
 'DATA_06MAST\\DATA_0064_mediana_bias.fits',
 'DATA_06MAST\\DATA_0067_mediana_bias.fits',
 'DATA_06MAST\\DATA_0072_mediana_bias.fits',
 'DATA_06MAST\\DATA_0073_mediana_bias.fits',
 'DATA_06MAST\\DATA_0074_mediana_bias.fits',
 'DATA_06MAST\\DATA_0077_mediana_bias.fits',
 'DATA_06MAST\\DATA_0078_mediana_bias.fits',
 'DATA_06MAST\\DATA_0079_mediana_bias.fits',
 'DATA_06MAST\\DATA_0081_mediana_bias.fits',
 'DATA_06MAST\\DATA_0082_mediana_bias.fits',
 'DATA_06MAST\\DATA_0088_mediana_bias.fits',
 'DATA_06MAST\\DATA_0090_mediana_bias.fits',
 'DATA_06MAST\\DATA_0094_mediana_bias.fits',
 'DATA_06MAST\\DATA_0101_mediana_bias.fits',
 'DATA_06MAST\\DATA_0104_mediana_bias.fits',
 'DATA_06MAST\\DATA_0109_mediana_bias.fits',
 'DATA_06MAST\\DATA_0123_mediana_bias.fits',
 'DATA_06M

In [115]:
ciencia_mediana_R

['DATA_06MAST\\DATA_0049_mediana_bias.fits',
 'DATA_06MAST\\DATA_0050_mediana_bias.fits',
 'DATA_06MAST\\DATA_0061_mediana_bias.fits',
 'DATA_06MAST\\DATA_0062_mediana_bias.fits',
 'DATA_06MAST\\DATA_0080_mediana_bias.fits',
 'DATA_06MAST\\DATA_0087_mediana_bias.fits',
 'DATA_06MAST\\DATA_0091_mediana_bias.fits',
 'DATA_06MAST\\DATA_0092_mediana_bias.fits',
 'DATA_06MAST\\DATA_0093_mediana_bias.fits',
 'DATA_06MAST\\DATA_0099_mediana_bias.fits',
 'DATA_06MAST\\DATA_0100_mediana_bias.fits',
 'DATA_06MAST\\DATA_0105_mediana_bias.fits',
 'DATA_06MAST\\DATA_0106_mediana_bias.fits',
 'DATA_06MAST\\DATA_0117_mediana_bias.fits',
 'DATA_06MAST\\DATA_0127_mediana_bias.fits',
 'DATA_06MAST\\DATA_0131_mediana_bias.fits',
 'DATA_06MAST\\DATA_0138_mediana_bias.fits',
 'DATA_06MAST\\DATA_0142_mediana_bias.fits',
 'DATA_06MAST\\DATA_0146_mediana_bias.fits',
 'DATA_06MAST\\DATA_0148_mediana_bias.fits',
 'DATA_06MAST\\DATA_0150_mediana_bias.fits',
 'DATA_06MAST\\DATA_0151_mediana_bias.fits',
 'DATA_06M

In [116]:
ciencia_mediana_I

['DATA_06MAST\\DATA_0059_mediana_bias.fits',
 'DATA_06MAST\\DATA_0066_mediana_bias.fits',
 'DATA_06MAST\\DATA_0069_mediana_bias.fits',
 'DATA_06MAST\\DATA_0070_mediana_bias.fits',
 'DATA_06MAST\\DATA_0071_mediana_bias.fits',
 'DATA_06MAST\\DATA_0075_mediana_bias.fits',
 'DATA_06MAST\\DATA_0076_mediana_bias.fits',
 'DATA_06MAST\\DATA_0083_mediana_bias.fits',
 'DATA_06MAST\\DATA_0084_mediana_bias.fits',
 'DATA_06MAST\\DATA_0085_mediana_bias.fits',
 'DATA_06MAST\\DATA_0096_mediana_bias.fits',
 'DATA_06MAST\\DATA_0102_mediana_bias.fits',
 'DATA_06MAST\\DATA_0103_mediana_bias.fits',
 'DATA_06MAST\\DATA_0108_mediana_bias.fits',
 'DATA_06MAST\\DATA_0112_mediana_bias.fits',
 'DATA_06MAST\\DATA_0114_mediana_bias.fits',
 'DATA_06MAST\\DATA_0115_mediana_bias.fits',
 'DATA_06MAST\\DATA_0116_mediana_bias.fits',
 'DATA_06MAST\\DATA_0118_mediana_bias.fits',
 'DATA_06MAST\\DATA_0124_mediana_bias.fits',
 'DATA_06MAST\\DATA_0125_mediana_bias.fits',
 'DATA_06MAST\\DATA_0126_mediana_bias.fits',
 'DATA_06M

In [118]:
#imágenes escogidas para cada filtro:
#para el filtro B: 0097
#para el filtro V: 0088
#para el filtro R: 0194
#para el filtro I: 0189

In [330]:
#visualización de las imágenes escogidas para extraer las coor. de las estrellas:
#solocambiar el número 'DATA_XXXX' por el número de la estrella escogida
imagen_sci0 = fits.getdata('DATA_06MAST/DATA_0097_mediana_bias_mediana_mediana_flat.fits')
#llamado a la función:
#visualizar_flat(imagen_sci0)
imagen_sci00 = fits.open('DATA_06MAST/DATA_0189_mediana_bias_mediana_mediana_flat.fits')
imagen_sci00[0].header


SIMPLE  =                    T / conforms to FITS standard                      
BITPIX  =                  -64 / array data type                                
NAXIS   =                    2 / number of array dimensions                     
NAXIS1  =                 2048                                                  
NAXIS2  =                 2108                                                  
DATE-OBS= '2021-07-05T00:26:10.75' /YYYY-MM-DDThh:mm:ss observation start, UT   
EXPTIME =   5.0000000000000000 /Exposure time in seconds                        
EXPOSURE=   5.0000000000000000 /Exposure time in seconds                        
SET-TEMP=  -90.000000000000000 /CCD temperature setpoint in C                   
CCD-TEMP=  -89.630996704101563 /CCD temperature at start of exposure in C       
XPIXSZ  =   13.500000000000000 /Pixel Width in microns (after binning)          
YPIXSZ  =   13.500000000000000 /Pixel Height in microns (after binning)         
XBINNING=                   

In [131]:
'''
Cálculo de la escala de la placa:
    De la cabecera de las imágenes se sabe que el tamaño del píxel de la cámara es de
        13.5 micras.
    De la cabecera tambián se sabe que la distancia focal es de 5271 mm
'''
size_pixel = 13.5
dist_focal = 5271

#conversión a mm:
# 1 mm = 1000 micras
size_pixel_mm = 13.5/1000

escala_placa = (206265 * size_pixel_mm)/(dist_focal)

print(f'La escala de la placa es {escala_placa} arcsec/pixel')
print('_' *20)

#calculo del seeing por filtro:

fwhm_B = 8.71
fwhm_V = 5.76
fwhm_R = 5.73
fwhm_I = 5.52

seeing_B = fwhm_B * escala_placa
seeing_V = fwhm_V * escala_placa
seeing_R = fwhm_R * escala_placa
seeing_I = fwhm_I * escala_placa

print(f'El seeing para el filtro B es {seeing_B} arcsec')
print(f'El seeing para el filtro V es {seeing_V} arcsec')
print(f'El seeing para el filtro R es {seeing_R} arcsec')
print(f'El seeing para el filtro I es {seeing_I} arcsec')


La escala de la placa es 0.5282825839499146 arcsec/pixel
____________________
El seeing para el filtro B es 4.601341306203756 arcsec
El seeing para el filtro V es 3.042907683551508 arcsec
El seeing para el filtro R es 3.0270592060330106 arcsec
El seeing para el filtro I es 2.9161198634035284 arcsec


**10.- Localiza entre 2 y 5 estrellas de comparación en SIMBAD para cada filtro e 
indica cuales son en la memoria, dando información sobre su nombre o 
identificación, su magnitud en B, V, R e I y sus colores B-V, V-R e V-I**

In [132]:
#listar los elementos de cada filtro: 

for elemento in ciencia_mediana_B:
    header = fits.getheader(elemento)
    print(elemento, header['OBJECT'])

DATA_06MAST\DATA_0051_mediana_bias.fits 4U0115
DATA_06MAST\DATA_0052_mediana_bias.fits CEPX4
DATA_06MAST\DATA_0056_mediana_bias.fits SA110-503
DATA_06MAST\DATA_0057_mediana_bias.fits SA112-250
DATA_06MAST\DATA_0060_mediana_bias.fits SAX2239
DATA_06MAST\DATA_0065_mediana_bias.fits IGR01583
DATA_06MAST\DATA_0068_mediana_bias.fits IGR22534
DATA_06MAST\DATA_0086_mediana_bias.fits SA110-233
DATA_06MAST\DATA_0089_mediana_bias.fits AX1841
DATA_06MAST\DATA_0095_mediana_bias.fits SAX2103
DATA_06MAST\DATA_0097_mediana_bias.fits SA110-503
DATA_06MAST\DATA_0098_mediana_bias.fits PG2331
DATA_06MAST\DATA_0107_mediana_bias.fits SA110-233
DATA_06MAST\DATA_0110_mediana_bias.fits IGR01363
DATA_06MAST\DATA_0111_mediana_bias.fits PG2331
DATA_06MAST\DATA_0113_mediana_bias.fits 4U2206
DATA_06MAST\DATA_0119_mediana_bias.fits AX1845
DATA_06MAST\DATA_0120_mediana_bias.fits SA110-503
DATA_06MAST\DATA_0121_mediana_bias.fits SWIFT0243
DATA_06MAST\DATA_0122_mediana_bias.fits MWC656
DATA_06MAST\DATA_0133_mediana_bi

In [137]:
#CAMPO UTILIZADO PARA INCISO 11 FILTRO B
#visualización de las imágenes escogidas para extraer las coor. de las estrellas:
#solocambiar el número 'DATA_XXXX' por el número de la estrella escogida
imagen_sci = fits.getdata('DATA_06MAST/DATA_0056_mediana_bias_mediana_mediana_flat.fits')
#llamado a la función:
visualizar_flat(imagen_sci)

In [133]:
#listar los elementos de cada filtro: 

for elemento in ciencia_mediana_V:
    header = fits.getheader(elemento)
    print(elemento, header['OBJECT'])

DATA_06MAST\DATA_0053_mediana_bias.fits PG2213
DATA_06MAST\DATA_0054_mediana_bias.fits 4U1907
DATA_06MAST\DATA_0055_mediana_bias.fits AX1841
DATA_06MAST\DATA_0058_mediana_bias.fits SA110-233
DATA_06MAST\DATA_0063_mediana_bias.fits SAX2103
DATA_06MAST\DATA_0064_mediana_bias.fits SA110-503
DATA_06MAST\DATA_0067_mediana_bias.fits KS1947
DATA_06MAST\DATA_0072_mediana_bias.fits AX1845
DATA_06MAST\DATA_0073_mediana_bias.fits SA112-250
DATA_06MAST\DATA_0074_mediana_bias.fits 4U0115
DATA_06MAST\DATA_0077_mediana_bias.fits 2S0114
DATA_06MAST\DATA_0078_mediana_bias.fits 4U2206
DATA_06MAST\DATA_0079_mediana_bias.fits SA110-503
DATA_06MAST\DATA_0081_mediana_bias.fits SAX2239
DATA_06MAST\DATA_0082_mediana_bias.fits XTE1855
DATA_06MAST\DATA_0088_mediana_bias.fits IGR01583
DATA_06MAST\DATA_0090_mediana_bias.fits LSI61235
DATA_06MAST\DATA_0094_mediana_bias.fits SA110-503
DATA_06MAST\DATA_0101_mediana_bias.fits IGR01363
DATA_06MAST\DATA_0104_mediana_bias.fits XTE1946
DATA_06MAST\DATA_0109_mediana_bias.

In [138]:
#CAMPO UTILIZADO PARA INCISO 11 FILTRO V
#visualización de las imágenes escogidas para extraer las coor. de las estrellas:
#solocambiar el número 'DATA_XXXX' por el número de la estrella escogida
imagen_sci = fits.getdata('DATA_06MAST/DATA_0053_mediana_bias_mediana_mediana_flat.fits')
#llamado a la función:
visualizar_flat(imagen_sci)

In [134]:
#listar los elementos de cada filtro: 

for elemento in ciencia_mediana_R:
    header = fits.getheader(elemento)
    print(elemento, header['OBJECT'])

DATA_06MAST\DATA_0049_mediana_bias.fits IGR22534
DATA_06MAST\DATA_0050_mediana_bias.fits SA112-250
DATA_06MAST\DATA_0061_mediana_bias.fits LSI61303
DATA_06MAST\DATA_0062_mediana_bias.fits IGR01363
DATA_06MAST\DATA_0080_mediana_bias.fits AX1845
DATA_06MAST\DATA_0087_mediana_bias.fits SA110-233
DATA_06MAST\DATA_0091_mediana_bias.fits SWIFT0243
DATA_06MAST\DATA_0092_mediana_bias.fits GRO2058
DATA_06MAST\DATA_0093_mediana_bias.fits XTE1946
DATA_06MAST\DATA_0099_mediana_bias.fits LSI61235
DATA_06MAST\DATA_0100_mediana_bias.fits PG2213
DATA_06MAST\DATA_0105_mediana_bias.fits SA110-503
DATA_06MAST\DATA_0106_mediana_bias.fits CEPX4
DATA_06MAST\DATA_0117_mediana_bias.fits MWC656
DATA_06MAST\DATA_0127_mediana_bias.fits XTE1855
DATA_06MAST\DATA_0131_mediana_bias.fits SA110-503
DATA_06MAST\DATA_0138_mediana_bias.fits IGR00370
DATA_06MAST\DATA_0142_mediana_bias.fits PG2213
DATA_06MAST\DATA_0146_mediana_bias.fits PG2331
DATA_06MAST\DATA_0148_mediana_bias.fits SA110-233
DATA_06MAST\DATA_0150_mediana_

In [141]:
#CAMPO UTILIZADO PARA INCISO 11 FILTRO R
#visualización de las imágenes escogidas para extraer las coor. de las estrellas:
#solocambiar el número 'DATA_XXXX' por el número de la estrella escogida
imagen_sci = fits.getdata('DATA_06MAST/DATA_0150_mediana_bias_mediana_mediana_flat.fits')
#llamado a la función:
visualizar_flat(imagen_sci)

In [135]:
#listar los elementos de cada filtro: 

for elemento in ciencia_mediana_I:
    header = fits.getheader(elemento)
    print(elemento, header['OBJECT'])

DATA_06MAST\DATA_0059_mediana_bias.fits PG2331
DATA_06MAST\DATA_0066_mediana_bias.fits SWIFT0243
DATA_06MAST\DATA_0069_mediana_bias.fits SA110-503
DATA_06MAST\DATA_0070_mediana_bias.fits PG2331
DATA_06MAST\DATA_0071_mediana_bias.fits GRO2058
DATA_06MAST\DATA_0075_mediana_bias.fits MWC656
DATA_06MAST\DATA_0076_mediana_bias.fits CEPX4
DATA_06MAST\DATA_0083_mediana_bias.fits SA112-250
DATA_06MAST\DATA_0084_mediana_bias.fits IGR01583
DATA_06MAST\DATA_0085_mediana_bias.fits XTE1855
DATA_06MAST\DATA_0096_mediana_bias.fits 4U2206
DATA_06MAST\DATA_0102_mediana_bias.fits 4U1907
DATA_06MAST\DATA_0103_mediana_bias.fits PG1633
DATA_06MAST\DATA_0108_mediana_bias.fits AX1845
DATA_06MAST\DATA_0112_mediana_bias.fits SAX2239
DATA_06MAST\DATA_0114_mediana_bias.fits SA110-503
DATA_06MAST\DATA_0115_mediana_bias.fits PG2213
DATA_06MAST\DATA_0116_mediana_bias.fits XTE1946
DATA_06MAST\DATA_0118_mediana_bias.fits 4U0115
DATA_06MAST\DATA_0124_mediana_bias.fits IGR22534
DATA_06MAST\DATA_0125_mediana_bias.fits S

In [142]:
#CAMPO UTILIZADO PARA INCISO 11 FILTRO I
#visualización de las imágenes escogidas para extraer las coor. de las estrellas:
#solocambiar el número 'DATA_XXXX' por el número de la estrella escogida
imagen_sci = fits.getdata('DATA_06MAST/DATA_0103_mediana_bias_mediana_mediana_flat.fits')
#llamado a la función:
visualizar_flat(imagen_sci)

In [143]:
'''
bias_media=np.average(datos_bias, axis=0)
bias_mediana=np.median(datos_bias, axis=0)

new_bias_media =
'''

'\nbias_media=np.average(datos_bias, axis=0)\nbias_mediana=np.median(datos_bias, axis=0)\n\nnew_bias_media =\n'

**SEGUNDA PARTE**

**0.- Describe brevemente (máximo 200 palabras) cómo se realiza fotometría de
abertura y porqué se observan estrellas estándar.**

In [5]:
#listar los .fits que hemos corregido (en este caso con la mediana para bias 
#y con la mediana para flats:

lista_bias_flat=glob.glob('DATA_06MAST/DATA_????_mediana_bias_mediana_mediana_flat.fits', 
                          recursive=True)
for elemento in lista_bias_flat:
    header = fits.getheader(elemento)
    print(elemento, header['OBJECT'])




DATA_06MAST\DATA_0049_mediana_bias_mediana_mediana_flat.fits IGR22534
DATA_06MAST\DATA_0050_mediana_bias_mediana_mediana_flat.fits SA112-250
DATA_06MAST\DATA_0051_mediana_bias_mediana_mediana_flat.fits 4U0115
DATA_06MAST\DATA_0052_mediana_bias_mediana_mediana_flat.fits CEPX4
DATA_06MAST\DATA_0053_mediana_bias_mediana_mediana_flat.fits PG2213
DATA_06MAST\DATA_0054_mediana_bias_mediana_mediana_flat.fits 4U1907
DATA_06MAST\DATA_0055_mediana_bias_mediana_mediana_flat.fits AX1841
DATA_06MAST\DATA_0056_mediana_bias_mediana_mediana_flat.fits SA110-503
DATA_06MAST\DATA_0057_mediana_bias_mediana_mediana_flat.fits SA112-250
DATA_06MAST\DATA_0058_mediana_bias_mediana_mediana_flat.fits SA110-233
DATA_06MAST\DATA_0059_mediana_bias_mediana_mediana_flat.fits PG2331
DATA_06MAST\DATA_0060_mediana_bias_mediana_mediana_flat.fits SAX2239
DATA_06MAST\DATA_0061_mediana_bias_mediana_mediana_flat.fits LSI61303
DATA_06MAST\DATA_0062_mediana_bias_mediana_mediana_flat.fits IGR01363
DATA_06MAST\DATA_0063_mediana_

**1.- Para dos estrellas estándares de un campo que elijas, representa en una gráfica,
para varias aperturas (al menos 10), la magnitud instrumental medida frente al
tamaño en píxeles de la abertura. En base a la gráfica obtenida justifica la elección
de un abertura adecuada para realizar la fotometría.**

In [20]:
#campo escogido: SA110 
#imagen 0056
#solocambiar el número 'DATA_XXXX' por el número de la estrella escogida
imagen_sci = fits.getdata('DATA_06MAST/DATA_0056_mediana_bias_mediana_mediana_flat.fits')
#llamado a la función:
visualizar_flat(imagen_sci)

In [6]:
%pip install photutils

Note: you may need to restart the kernel to use updated packages.


In [13]:
header = fits.getheader('DATA_06MAST/DATA_0056_mediana_bias_mediana_mediana_flat.fits')
gain = header['GAIN']
r_noise = header['RDNOISE']
t_exp = header['EXPTIME']
print(gain)
print(r_noise)
print(t_exp)

0.865
5.95
20.0


In [7]:
from astropy.stats import sigma_clipped_stats

In [21]:
main, mediana, std = sigma_clipped_stats(imagen_sci, sigma = 3.0)

In [22]:
main

22.044669837106014

In [23]:
std

8.019450202185295

In [25]:
#leer las posiciones de las estrellas estándares que hemos escogido:
posiciones = np.loadtxt('DATA_06MAST/C0056.dat')
posiciones

array([[ 626., 1107.],
       [1022.,  979.]])

In [20]:
#Recentrar las coordenadas de las estrellas estándares 
#(para esto se usa photutils)

#para definir el anillo y la apertura alrededor de la estrella:
from photutils.aperture import CircularAperture, CircularAnnulus

#para recentar las posiciones:
from photutils.centroids import centroid_com, centroid_sources

#libreria que hace la fotometría de apertura:
from photutils.aperture import aperture_photometry

#librería para el cálculo de errores:
from photutils.utils import calc_total_error

In [41]:
#encontrar los centroides de las posiciones:
coor_x, coor_y = centroid_sources(imagen_sci, posiciones[:,0], posiciones[0:,1], 
                                  box_size = 5, centroid_func = centroid_com)
print(coor_x)
print(coor_y)

[ 626.60096433 1022.10708895]
[1107.58887324  978.95808369]


In [42]:
#agrupados las pocisiones corregidas como parejas:
nuevas_posiciones = np.transpose((coor_x, coor_y))
print(nuevas_posiciones)

[[ 626.60096433 1107.58887324]
 [1022.10708895  978.95808369]]


In [76]:
#crear un rango de valores de apertura:
#como una idea, una buena apertura puede estar entre 2 y 3 veces FWHM
#en este caso el FWHM es de 4.98
radios = np.arange(1,18,1)
radios

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17])

In [77]:
#crear las apreturas para cada radio:
aperturas = [CircularAperture(nuevas_posiciones, r = r) for r in radios]
aperturas


[<CircularAperture([[ 626.60096433, 1107.58887324],
                  [1022.10708895,  978.95808369]], r=1.0)>,
 <CircularAperture([[ 626.60096433, 1107.58887324],
                  [1022.10708895,  978.95808369]], r=2.0)>,
 <CircularAperture([[ 626.60096433, 1107.58887324],
                  [1022.10708895,  978.95808369]], r=3.0)>,
 <CircularAperture([[ 626.60096433, 1107.58887324],
                  [1022.10708895,  978.95808369]], r=4.0)>,
 <CircularAperture([[ 626.60096433, 1107.58887324],
                  [1022.10708895,  978.95808369]], r=5.0)>,
 <CircularAperture([[ 626.60096433, 1107.58887324],
                  [1022.10708895,  978.95808369]], r=6.0)>,
 <CircularAperture([[ 626.60096433, 1107.58887324],
                  [1022.10708895,  978.95808369]], r=7.0)>,
 <CircularAperture([[ 626.60096433, 1107.58887324],
                  [1022.10708895,  978.95808369]], r=8.0)>,
 <CircularAperture([[ 626.60096433, 1107.58887324],
                  [1022.10708895,  978.95808369]], r

In [78]:
#valor medio del cielo por pixel:
main

22.044669837106014

In [79]:
#para saber cuántos pixeles hay en cada apertura se debe calcular el área, y multiplicar
#esa área por el número de cuentas por pixel
#de esta forma se tiene todas las cuentas del cielo encerradas en el radio de apertura,
#luego se resta a las cuentas para obtener la magnitud instrumental. 
cielo = []
for i in range(17):
    cielo.append(aperturas[i].area*main)
cielo

[69.25537281106476,
 277.02149124425904,
 623.2983552995828,
 1108.0859649770362,
 1731.3843202766188,
 2493.193421198331,
 3393.5132677421725,
 4432.343859908145,
 5609.685197696245,
 6925.537281106475,
 8379.900110138835,
 9972.773684793325,
 11704.158005069943,
 13574.05307096869,
 15582.458882489569,
 17729.37543963258,
 20014.802742397715]

In [80]:
#tabla de fotometría:
ft = aperture_photometry(imagen_sci, aperturas) 
ft
#estos valores son intensidades

id,xcenter,ycenter,aperture_sum_0,aperture_sum_1,aperture_sum_2,aperture_sum_3,aperture_sum_4,aperture_sum_5,aperture_sum_6,aperture_sum_7,aperture_sum_8,aperture_sum_9,aperture_sum_10,aperture_sum_11,aperture_sum_12,aperture_sum_13,aperture_sum_14,aperture_sum_15,aperture_sum_16
,pix,pix,,,,,,,,,,,,,,,,,
int32,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64
1,626.6009643310948,1107.5888732406352,34507.06603455753,131580.82476428073,261613.61560052616,378182.3298457638,454742.9059179139,496543.7707563158,518671.9937894989,531245.2722349224,539605.5169233294,545453.4104559462,550113.1535465177,554298.0597673024,557965.7466939724,561431.9812406506,564887.5765864691,568367.6400546911,571647.1786033663
2,1022.107088946584,978.9580836912723,11266.713138278526,37280.36045355693,64525.74767855731,85124.70927564445,98612.8524489806,106968.26072891479,112226.92443891654,115777.2868728504,118610.64168981553,121101.83190220248,123504.68991707798,125761.22649414721,127883.93003929971,130164.47940098407,132505.08171968124,134995.22356455124,137575.8370041854


In [57]:
#para ver el tiempo de exposición de la imagen:
header

SIMPLE  =                    T / conforms to FITS standard                      
BITPIX  =                  -64 / array data type                                
NAXIS   =                    2 / number of array dimensions                     
NAXIS1  =                 2048                                                  
NAXIS2  =                 2108                                                  
DATE-OBS= '2021-07-04T20:50:09.54' /YYYY-MM-DDThh:mm:ss observation start, UT   
EXPTIME =   20.000000000000000 /Exposure time in seconds                        
EXPOSURE=   20.000000000000000 /Exposure time in seconds                        
SET-TEMP=  -90.000000000000000 /CCD temperature setpoint in C                   
CCD-TEMP=  -89.630996704101563 /CCD temperature at start of exposure in C       
XPIXSZ  =   13.500000000000000 /Pixel Width in microns (after binning)          
YPIXSZ  =   13.500000000000000 /Pixel Height in microns (after binning)         
XBINNING=                   

In [84]:
#como se ve en la cabecera, el tiempo de exposición es de 20 segundos
#ahora de agregara las magnitudes instrumentales para cada apertura a la tabla de intensidades:
#para la constante usaremos '20':
for i in range(17):
    ft[f'mag_{i}'] = -2.5*np.log10((ft[f'aperture_sum_{i}']-cielo[i])/20)+20

In [85]:
ft

id,xcenter,ycenter,aperture_sum_0,aperture_sum_1,aperture_sum_2,aperture_sum_3,aperture_sum_4,aperture_sum_5,aperture_sum_6,aperture_sum_7,aperture_sum_8,aperture_sum_9,aperture_sum_10,aperture_sum_11,aperture_sum_12,aperture_sum_13,aperture_sum_14,aperture_sum_15,aperture_sum_16,mag_0,mag_1,mag_2,mag_3,mag_4,mag_5,mag_6,mag_7,mag_8,mag_9,mag_10,mag_11,mag_12,mag_13,mag_14,mag_15,mag_16
,pix,pix,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
int32,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64
1,626.6009643310948,1107.5888732406352,34507.06603455753,131580.82476428073,261613.61560052616,378182.3298457638,454742.9059179139,496543.7707563158,518671.9937894989,531245.2722349224,539605.5169233294,545453.4104559462,550113.1535465177,554298.0597673024,557965.7466939724,561431.9812406506,564887.5765864691,568367.6400546911,571647.1786033663,11.90998615425479,10.456881724648916,9.711014000938508,9.311507817006037,9.112301869874866,9.018146461318548,8.97246997545732,8.948433928347159,8.933730321614881,8.924554522986973,8.918111251951814,8.912928715414719,8.909073330574138,8.905905112161971,8.903040876094792,8.90040901958355,8.898450620576545
2,1022.107088946584,978.9580836912723,11266.713138278526,37280.36045355693,64525.74767855731,85124.70927564445,98612.8524489806,106968.26072891479,112226.92443891654,115777.2868728504,118610.64168981553,121101.83190220248,123504.68991707798,125761.22649414721,127883.93003929971,130164.47940098407,132505.08171968124,134995.22356455124,137575.8370041854,13.129776406536486,11.831972704162613,11.238781227676743,10.94166193180764,10.786973210597758,10.705043340138612,10.660669385410717,10.635898745825665,10.61986969024142,10.60863512745709,10.599652863466252,10.593411862081007,10.589748689218561,10.585917763247126,10.582828616670218,10.579646116809675,10.57691649368056


In [87]:
#curva de crecimiento:
for i in range(1): 
    mags = [ft[f'mag_{j}'] for j in range(17)]
    plt.plot(radios, mags, label = f'Estrella {i}')

#etiqueta del eje x:
plt.xlabel('Apertura_0')

#etiqueta del eje y:
plt.ylabel('Magnitud Instrumental')

#grid:
plt.grid(True, alpha = 0.8)

#legenda:
plt.legend()

In [95]:
from matplotlib.ticker import MultipleLocator #para controlar el espaciado de los ejes

marcadores = ['o', 's']
colores = ['navy', 'darkred']

plt.figure(figsize=(9, 6))

#Bucle para graficar todas las estrellas de la tabla ft
for i in range(len(ft)):
    mags = [ft[i][f'mag_{j}'] for j in range(17)]
    color_estrella = colores[i % len(colores)]
    estilo_marcador = marcadores[i % len(marcadores)]
    plt.plot(radios, mags, color=color_estrella, marker=estilo_marcador, markersize=5, 
             label=f'Estrella {i+1}')

#eje y a cada 0.5 unidades
plt.gca().yaxis.set_major_locator(MultipleLocator(0.5))

#marcas pequeñas (minor ticks) cada 0.1 para mayor precisión visual
#plt.gca().yaxis.set_minor_locator(MultipleLocator(0.1))

#Invertir eje y:
plt.gca().invert_yaxis()

plt.xlabel('Radio de Apertura (píxeles)', fontsize=12)
plt.ylabel('Magnitud Instrumental', fontsize=12)
plt.title('Curva de Crecimiento Fotométrica', fontsize=14, fontweight='bold')

plt.grid(which='major', linestyle='-', alpha=0.5)
#plt.grid(which='minor', linestyle=':', alpha=0.3)

plt.legend(loc='best', frameon=True, shadow=False)
plt.tight_layout()

plt.show()

Respuesta inciso 1:
Para la creación de la curva de crecimieto se seleccionaron dos estrellas estádar del campo SA110 (específicamente el fichero llamado 'DATA_0056_mediana_bias_mediana_mediana_flat.fits').
A partir de la curva de intensidad de estas estrellas, se obtuvo un FWHM promedio de 4.98 píxeles, lo que nos da una idea de que una buena apertura está entre 9 y 14 píxeles (entre 2 y 3 veces el FWHM). 
La gráfica de crecimiento se construyó con un rango de aperturas entre 1 y 16 píxeles. En esta gráfica para las dos estellas seleccionadas podemos destacar el siguiente comportamiento: 
Para radios <8 píxeles: La pendiente de la curva es pronunciada, que indica que hay una pérdida de flujo significativa.
Para radios entre 8 y 12 píxeles: La curva comienza a aplanarse, que indica que ya se ha capturado casi toda la luz de la estrella.
Para radios > 12 píxeles: El valor de la magnitud es casi constante, que indica que ya se ha capturado la totalidad del flujo de la estrella. 

**Considerando lo anterior se toma un radio de apertura de 12 píxeles, pues con este radio ya se ha capturado el flujo de la estrella y no es demasiado grande para añadir ruido de fondo innecesaio.**

**2.- Haz una tabla por filtro que contenga, para todas las estrellas estándar:**

In [9]:
#organizar los archivos corregidos por bias y flat por filtro:
corregidos_B = []
corregidos_V = []
corregidos_R = []
corregidos_I = []

for elemento in lista_bias_flat:
    header = fits.getheader(elemento)
    filtro = header['FILTER']
    if filtro == 'B':
        corregidos_B.append(elemento)
    elif filtro == 'V':
        corregidos_V.append(elemento)
    elif filtro == 'R':
        corregidos_R.append(elemento)
    else:
        corregidos_I.append(elemento)


In [10]:
for elemento in corregidos_B:
    header = fits.getheader(elemento)
    print(elemento, header['FILTER'], header['OBJECT'])

len(corregidos_B)

DATA_06MAST\DATA_0051_mediana_bias_mediana_mediana_flat.fits B 4U0115
DATA_06MAST\DATA_0052_mediana_bias_mediana_mediana_flat.fits B CEPX4
DATA_06MAST\DATA_0056_mediana_bias_mediana_mediana_flat.fits B SA110-503
DATA_06MAST\DATA_0057_mediana_bias_mediana_mediana_flat.fits B SA112-250
DATA_06MAST\DATA_0060_mediana_bias_mediana_mediana_flat.fits B SAX2239
DATA_06MAST\DATA_0065_mediana_bias_mediana_mediana_flat.fits B IGR01583
DATA_06MAST\DATA_0068_mediana_bias_mediana_mediana_flat.fits B IGR22534
DATA_06MAST\DATA_0086_mediana_bias_mediana_mediana_flat.fits B SA110-233
DATA_06MAST\DATA_0089_mediana_bias_mediana_mediana_flat.fits B AX1841
DATA_06MAST\DATA_0095_mediana_bias_mediana_mediana_flat.fits B SAX2103
DATA_06MAST\DATA_0097_mediana_bias_mediana_mediana_flat.fits B SA110-503
DATA_06MAST\DATA_0098_mediana_bias_mediana_mediana_flat.fits B PG2331
DATA_06MAST\DATA_0107_mediana_bias_mediana_mediana_flat.fits B SA110-233
DATA_06MAST\DATA_0110_mediana_bias_mediana_mediana_flat.fits B IGR0136

39

In [107]:
for elemento in corregidos_V:
    header = fits.getheader(elemento)
    print(elemento, header['FILTER'], header['OBJECT'])

len(corregidos_V)

DATA_06MAST\DATA_0053_mediana_bias_mediana_mediana_flat.fits V PG2213
DATA_06MAST\DATA_0054_mediana_bias_mediana_mediana_flat.fits V 4U1907
DATA_06MAST\DATA_0055_mediana_bias_mediana_mediana_flat.fits V AX1841
DATA_06MAST\DATA_0058_mediana_bias_mediana_mediana_flat.fits V SA110-233
DATA_06MAST\DATA_0063_mediana_bias_mediana_mediana_flat.fits V SAX2103
DATA_06MAST\DATA_0064_mediana_bias_mediana_mediana_flat.fits V SA110-503
DATA_06MAST\DATA_0067_mediana_bias_mediana_mediana_flat.fits V KS1947
DATA_06MAST\DATA_0072_mediana_bias_mediana_mediana_flat.fits V AX1845
DATA_06MAST\DATA_0073_mediana_bias_mediana_mediana_flat.fits V SA112-250
DATA_06MAST\DATA_0074_mediana_bias_mediana_mediana_flat.fits V 4U0115
DATA_06MAST\DATA_0077_mediana_bias_mediana_mediana_flat.fits V 2S0114
DATA_06MAST\DATA_0078_mediana_bias_mediana_mediana_flat.fits V 4U2206
DATA_06MAST\DATA_0079_mediana_bias_mediana_mediana_flat.fits V SA110-503
DATA_06MAST\DATA_0081_mediana_bias_mediana_mediana_flat.fits V SAX2239
DATA_0

39

In [108]:
for elemento in corregidos_R:
    header = fits.getheader(elemento)
    print(elemento, header['FILTER'], header['OBJECT'])

len(corregidos_R)

DATA_06MAST\DATA_0049_mediana_bias_mediana_mediana_flat.fits R IGR22534
DATA_06MAST\DATA_0050_mediana_bias_mediana_mediana_flat.fits R SA112-250
DATA_06MAST\DATA_0061_mediana_bias_mediana_mediana_flat.fits R LSI61303
DATA_06MAST\DATA_0062_mediana_bias_mediana_mediana_flat.fits R IGR01363
DATA_06MAST\DATA_0080_mediana_bias_mediana_mediana_flat.fits R AX1845
DATA_06MAST\DATA_0087_mediana_bias_mediana_mediana_flat.fits R SA110-233
DATA_06MAST\DATA_0091_mediana_bias_mediana_mediana_flat.fits R SWIFT0243
DATA_06MAST\DATA_0092_mediana_bias_mediana_mediana_flat.fits R GRO2058
DATA_06MAST\DATA_0093_mediana_bias_mediana_mediana_flat.fits R XTE1946
DATA_06MAST\DATA_0099_mediana_bias_mediana_mediana_flat.fits R LSI61235
DATA_06MAST\DATA_0100_mediana_bias_mediana_mediana_flat.fits R PG2213
DATA_06MAST\DATA_0105_mediana_bias_mediana_mediana_flat.fits R SA110-503
DATA_06MAST\DATA_0106_mediana_bias_mediana_mediana_flat.fits R CEPX4
DATA_06MAST\DATA_0117_mediana_bias_mediana_mediana_flat.fits R MWC656

39

In [109]:
for elemento in corregidos_I:
    header = fits.getheader(elemento)
    print(elemento, header['FILTER'], header['OBJECT'])

len(corregidos_I)

DATA_06MAST\DATA_0059_mediana_bias_mediana_mediana_flat.fits I PG2331
DATA_06MAST\DATA_0066_mediana_bias_mediana_mediana_flat.fits I SWIFT0243
DATA_06MAST\DATA_0069_mediana_bias_mediana_mediana_flat.fits I SA110-503
DATA_06MAST\DATA_0070_mediana_bias_mediana_mediana_flat.fits I PG2331
DATA_06MAST\DATA_0071_mediana_bias_mediana_mediana_flat.fits I GRO2058
DATA_06MAST\DATA_0075_mediana_bias_mediana_mediana_flat.fits I MWC656
DATA_06MAST\DATA_0076_mediana_bias_mediana_mediana_flat.fits I CEPX4
DATA_06MAST\DATA_0083_mediana_bias_mediana_mediana_flat.fits I SA112-250
DATA_06MAST\DATA_0084_mediana_bias_mediana_mediana_flat.fits I IGR01583
DATA_06MAST\DATA_0085_mediana_bias_mediana_mediana_flat.fits I XTE1855
DATA_06MAST\DATA_0096_mediana_bias_mediana_mediana_flat.fits I 4U2206
DATA_06MAST\DATA_0102_mediana_bias_mediana_mediana_flat.fits I 4U1907
DATA_06MAST\DATA_0103_mediana_bias_mediana_mediana_flat.fits I PG1633
DATA_06MAST\DATA_0108_mediana_bias_mediana_mediana_flat.fits I AX1845
DATA_06M

39

In [46]:
#visualizar el fichero corregido:
#solo cambiar el número 'DATA_XXXX' por el número de la estrella escogida
ima = 'DATA_06MAST/DATA_0056_mediana_bias_mediana_mediana_flat.fits'
imagen_sci = fits.getdata(ima)
#llamado a la función:
visualizar_flat(imagen_sci)

In [30]:
gain = header['GAIN']
r_noise = header['RDNOISE']
t_exp = header['EXPTIME']
print(gain)
print(r_noise)
print(t_exp)

0.865
5.95
8.0


In [45]:
#leer las coordenadas
archivo_coor = 'CE_0056_B.dat'
posiciones = np.loadtxt(archivo_coor)
posiciones

array([[1685.,  765.,  496.],
       [1508.,  805.,  497.],
       [1229., 1421.,  499.],
       [1098., 1489.,  502.],
       [1018., 1061.,  503.],
       [1018.,  987.,  504.],
       [ 636.,  902.,  506.],
       [ 624., 1113.,  507.]])

In [23]:
#recentrar:
coor_x, coor_y = centroid_sources(imagen_sci, posiciones[:,0], posiciones[:,1],
                                 box_size = 5, centroid_func = centroid_com)
coor_x
coor_y

array([ 763.9877548 ,  804.20584301, 1419.90446383, 1488.36222447,
       1060.16893802,  986.6450694 ,  901.46313976, 1111.89402294])

In [24]:
#organizar las coordenadas en pares:
posiciones_centradas = np.transpose((coor_x, coor_y))
posiciones_centradas

array([[1684.70922111,  763.9877548 ],
       [1508.54784541,  804.20584301],
       [1229.31345264, 1419.90446383],
       [1098.18998997, 1488.36222447],
       [1018.39224293, 1060.16893802],
       [1018.12097845,  986.6450694 ],
       [ 636.17825782,  901.46313976],
       [ 624.82629976, 1111.89402294]])

In [27]:
#preparar la apertura:
apertura = [CircularAperture(posiciones_centradas, r = 12)]

#anillo:
anillo = [CircularAnnulus(posiciones_centradas,r_in=20, r_out=25)]

#apertura total:
full_ap = apertura +anillo

full_ap

[<CircularAperture([[1684.70922111,  763.9877548 ],
                  [1508.54784541,  804.20584301],
                  [1229.31345264, 1419.90446383],
                  [1098.18998997, 1488.36222447],
                  [1018.39224293, 1060.16893802],
                  [1018.12097845,  986.6450694 ],
                  [ 636.17825782,  901.46313976],
                  [ 624.82629976, 1111.89402294]], r=12.0)>,
 <CircularAnnulus([[1684.70922111,  763.9877548 ],
                 [1508.54784541,  804.20584301],
                 [1229.31345264, 1419.90446383],
                 [1098.18998997, 1488.36222447],
                 [1018.39224293, 1060.16893802],
                 [1018.12097845,  986.6450694 ],
                 [ 636.17825782,  901.46313976],
                 [ 624.82629976, 1111.89402294]], r_in=20.0, r_out=25.0)>]

In [33]:
#cálculo de desviación estándar para cada píxel:
error_map = calc_total_error(imagen_sci, bkg_error = r_noise, effective_gain = gain)
error_map

array([[   5.95      ,  233.33328249,    5.95      , ...,  780.89102324,
           5.95      ,   59.37906788],
       [ 135.72692582,    5.95      ,  362.45057532, ...,  123.09875984,
           5.95      ,  118.3101319 ],
       [ 406.83284718,  430.14656158,  201.48137879, ...,  333.0165438 ,
           5.95      ,    5.95      ],
       ...,
       [ 148.65758498,  276.05801565,  121.59052053, ...,   78.70544981,
        1069.4871953 ,   78.38193098],
       [   5.95      ,  177.58198224,    5.95      , ...,  166.58798422,
           5.95      ,   72.60240563],
       [ 259.71558953,    5.95      ,    5.95      , ...,  215.00928463,
           5.95      ,    5.95      ]])

In [34]:
#fotometría de apertura:
fa = aperture_photometry(imagen_sci, full_ap, error = error_map)
fa

id,xcenter,ycenter,aperture_sum_0,aperture_sum_err_0,aperture_sum_1,aperture_sum_err_1
,pix,pix,,,,
int32,float64,float64,float64,float64,float64,float64
1,1684.709221112876,763.9877547999131,350228.5917358973,648.771345732619,17990.142779377216,214.06168633327604
2,1508.547845406621,804.2058430140066,125459.93895141463,401.3179606442266,16393.382181534154,209.70560208725647
3,1229.3134526417618,1419.9044638254168,1183150.9227634498,1176.3588945007139,24021.87094773459,229.77271749325655
4,1098.1899899748184,1488.362224468413,240363.78184511207,542.1189151187319,17869.98557815137,213.73830876197601
5,1018.3922429306609,1060.1689380246937,1357134.611041523,1258.9509365154415,26604.524614945665,236.1804998028814
6,1018.12097845194,986.6450693995605,112054.91708031044,381.5219134263953,16805.232602795317,210.84783973441424
7,636.1782578232845,901.4631397608089,1438938.0080252455,1295.9659955764112,41334.92654965162,269.8343841797091
8,624.826299758798,1111.894022937528,549785.2916148112,807.2209401316139,19002.213065202995,216.7956006087458


In [36]:
#cielo/px:
cielo = fa['aperture_sum_1']/anillo[0].area
cielo

25.450845780149802
23.19189162831786
33.98399558797825
25.28085811327708
37.637703120698724
23.774540787057358
58.476958957488684
26.88263234677736


In [39]:
#cálculo de magnitud instrumental:
fa['mag_inst'] = -2.5*np.log10((fa['aperture_sum_0']-cielo*apertura[0].area)/t_exp)+20
fa

id,xcenter,ycenter,aperture_sum_0,aperture_sum_err_0,aperture_sum_1,aperture_sum_err_1,mag_inst
,pix,pix,,,,,
int32,float64,float64,float64,float64,float64,float64,float64
1,1684.709221112876,763.9877547999131,350228.5917358973,648.771345732619,17990.142779377216,214.06168633327604,8.433139212356862
2,1508.547845406621,804.2058430140066,125459.93895141463,401.3179606442266,16393.382181534154,209.70560208725647,9.606280879961046
3,1229.3134526417618,1419.9044638254168,1183150.9227634498,1176.3588945007139,24021.87094773459,229.77271749325655,7.0893252435423815
4,1098.1899899748184,1488.362224468413,240363.78184511207,542.1189151187319,17869.98557815137,213.73830876197601,8.85848246718367
5,1018.3922429306609,1060.1689380246937,1357134.611041523,1258.9509365154415,26604.524614945665,236.1804998028814,6.939875698666578
6,1018.12097845194,986.6450693995605,112054.91708031044,381.5219134263953,16805.232602795317,210.84783973441424,9.743705981977735
7,636.1782578232845,901.4631397608089,1438938.0080252455,1295.9659955764112,41334.92654965162,269.8343841797091,6.8827663900916
8,624.826299758798,1111.894022937528,549785.2916148112,807.2209401316139,19002.213065202995,216.7956006087458,7.931528600008534


In [44]:
#error de la magnitud instrumental:
fa['mag_inst_err'] = 1.0857*np.sqrt(fa['aperture_sum_err_0']**2
                                    +((fa['aperture_sum_1']/anillo[0].area)*apertura[0].area)**2)/fa['aperture_sum_0']
fa

id,xcenter,ycenter,aperture_sum_0,aperture_sum_err_0,aperture_sum_1,aperture_sum_err_1,mag_inst,mag_inst_err
,pix,pix,,,,,,
int32,float64,float64,float64,float64,float64,float64,float64,float64
1,1684.709221112876,763.9877547999131,350228.5917358973,648.771345732619,17990.142779377216,214.06168633327604,8.433139212356862,0.035748777177888984
2,1508.547845406621,804.2058430140066,125459.93895141463,401.3179606442266,16393.382181534154,209.70560208725647,9.606280879961046,0.09085959237679572
3,1229.3134526417618,1419.9044638254168,1183150.9227634498,1176.3588945007139,24021.87094773459,229.77271749325655,7.0893252435423815,0.014148947119766981
4,1098.1899899748184,1488.362224468413,240363.78184511207,542.1189151187319,17869.98557815137,213.73830876197601,8.85848246718367,0.051716883109204706
5,1018.3922429306609,1060.1689380246937,1357134.611041523,1258.9509365154415,26604.524614945665,236.1804998028814,6.939875698666578,0.013658603331230154
6,1018.12097845194,986.6450693995605,112054.91708031044,381.5219134263953,16805.232602795317,210.84783973441424,9.743705981977735,0.10427410944873527
7,636.1782578232845,901.4631397608089,1438938.0080252455,1295.9659955764112,41334.92654965162,269.8343841797091,6.8827663900916,0.019984137305044045
8,624.826299758798,1111.894022937528,549785.2916148112,807.2209401316139,19002.213065202995,216.7956006087458,7.931528600008534,0.02406885696545979


In [144]:
#visualizar la imagen
#ima2 = 'DATA_06MAST/DATA_0103_mediana_bias_mediana_mediana_flat.fits'
imagen_sci2 = fits.getdata(ima2)
#llamado a la función:
visualizar_flat(imagen_sci2)

In [244]:
#cambiar la imagen y las coordenadas:
ima = 'DATA_06MAST/DATA_0103_mediana_bias_mediana_mediana_flat.fits'
archivo_coor = 'CE_0103_I.dat'

#leer la imagen:
imagen_sci = fits.getdata(ima)
header = fits.getheader(ima)
#gain = header['GAIN']
#r_noise = header['RDNOISE']
#t_exp = header['EXPTIME']
#air_mass = header['AIRMASS']

#leer las coordenadas
posiciones = np.loadtxt(archivo_coor)
posiciones

#recentrar:
coor_x, coor_y = centroid_sources(imagen_sci, posiciones[:,0], posiciones[:,1],
                                 box_size = 5, centroid_func = centroid_com)

#organizar las coordenadas en pares:
posiciones_centradas = np.transpose((coor_x, coor_y))

#preparar la apertura:
apertura = [CircularAperture(posiciones_centradas, r = 12)]

#anillo:
anillo = [CircularAnnulus(posiciones_centradas,r_in=20, r_out=25)]

#apertura total:
full_ap = apertura + anillo

#cálculo de desviación estándar para cada píxel:
error_map = calc_total_error(imagen_sci, bkg_error = header['RDNOISE'], effective_gain = header['GAIN'])

#fotometría de apertura:
fa = aperture_photometry(imagen_sci, full_ap, error = error_map)

#cielo/px:
cielo = fa['aperture_sum_1']/anillo[0].area

#cálculo de magnitud instrumental:
fa['mag_inst'] = -2.5*np.log10((fa['aperture_sum_0']-cielo*apertura[0].area)/header['EXPTIME'])+20

#error de la magnitud instrumental:
fa['mag_inst_err'] = 1.0857*np.sqrt(fa['aperture_sum_err_0']**2
                                    +((fa['aperture_sum_1']/anillo[0].area)*apertura[0].area)**2)/fa['aperture_sum_0']

#columna de masa de aire
fa['Masa_aire'] = header['AIRMASS']

#exportar la tabla:
nombre_tabla = f"mag_instr_{archivo_coor}.csv"
fa.write(nombre_tabla, format='ascii.csv', delimiter=';', overwrite=True)

In [309]:
for elemento in corregidos_I:
    header2 = fits.getheader(elemento)
    print(elemento, header2['FILTER'], header2['OBJECT'], header2['AIRMASS'])


DATA_06MAST\DATA_0059_mediana_bias_mediana_mediana_flat.fits I PG2331 1.1684210573323688
DATA_06MAST\DATA_0066_mediana_bias_mediana_mediana_flat.fits I SWIFT0243 1.78710039692541
DATA_06MAST\DATA_0069_mediana_bias_mediana_mediana_flat.fits I SA110-503 1.2530001012592722
DATA_06MAST\DATA_0070_mediana_bias_mediana_mediana_flat.fits I PG2331 1.8565412414212832
DATA_06MAST\DATA_0071_mediana_bias_mediana_mediana_flat.fits I GRO2058 1.243508610570128
DATA_06MAST\DATA_0075_mediana_bias_mediana_mediana_flat.fits I MWC656 1.4761225688671098
DATA_06MAST\DATA_0076_mediana_bias_mediana_mediana_flat.fits I CEPX4 1.3163376683652082
DATA_06MAST\DATA_0083_mediana_bias_mediana_mediana_flat.fits I SA112-250 1.2437628456048722
DATA_06MAST\DATA_0084_mediana_bias_mediana_mediana_flat.fits I IGR01583 1.6040507524462115
DATA_06MAST\DATA_0085_mediana_bias_mediana_mediana_flat.fits I XTE1855 1.7396968419614443
DATA_06MAST\DATA_0096_mediana_bias_mediana_mediana_flat.fits I 4U2206 1.3859072426933938
DATA_06MAST\

**3.- Estima el coeficiente de extinción para cada filtro usando los datos de las tablas
anteriores.**

In [299]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from matplotlib.ticker import MultipleLocator

datos = np.loadtxt('ee.txt')
diff_mags = datos[:, 0]
mag_err = np.abs(datos[:, 1])
airmass = datos[:, 2]

mascara = (diff_mags > 3.8)

airmass_f = airmass[mascara]
diff_f = diff_mags[mascara]
err_f = mag_err[mascara]

def modelo_extincion(x, k, ZP):
    return ZP - k * x

popt, pcov = curve_fit(modelo_extincion, airmass_f, diff_f, sigma=err_f)
k_val, zp_val = popt
err_k = np.sqrt(pcov[0,0])
err_zp = np.sqrt(pcov[1,1])

plt.figure(figsize=(10, 6))

plt.errorbar(airmass[~mascara], diff_mags[~mascara], yerr=mag_err[~mascara], 
             fmt='o', color='lightgray', label='Descartados', alpha=0.5)

plt.errorbar(airmass_f, diff_f, yerr=err_f, 
             fmt='o', color='darkblue', ecolor='gray', capsize=3, label='Usados para el ajuste')

x_plot = np.linspace(1.0, 2.2, 100)
y_plot = modelo_extincion(x_plot, k_val, zp_val)
plt.plot(x_plot, y_plot, 'r--', label=f'Ajuste: k = {k_val:.3f} ± {err_k:.3f}')

plt.gca().invert_yaxis() # Invertir eje Y (Magnitudes)
plt.gca().yaxis.set_major_locator(MultipleLocator(0.2)) # Marcas cada 0.2 mag
plt.xlabel('Masa de Aire (X)')
plt.ylabel('$m_{std} - m_{inst}$')
plt.title('Solución Fotométrica (Filtro B)')
plt.legend()
plt.grid(alpha=0.2)

plt.show()

print(f"Resultado: k = {k_val:.4f} ± {err_k:.4f} mag/airmass")
print(f"Término independiente (ZP): {zp_val:.4f} ± {err_zp:.4f} mag")

Resultado: k = 0.3621 ± 0.6945 mag/airmass
Término independiente (ZP): 4.9196 ± 0.8835 mag


In [302]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from matplotlib.ticker import MultipleLocator

datos = np.loadtxt('vvv.txt')
diff_mags = datos[:, 0]
mag_err = np.abs(datos[:, 1])
airmass = datos[:, 2]

mascara = (diff_mags > 3.8)

airmass_f = airmass[mascara]
diff_f = diff_mags[mascara]
err_f = mag_err[mascara]

def modelo_extincion(x, k, ZP):
    return ZP - k * x

popt, pcov = curve_fit(modelo_extincion, airmass_f, diff_f, sigma=err_f)
k_val, zp_val = popt
err_k = np.sqrt(pcov[0,0])
err_zp = np.sqrt(pcov[1,1])

plt.figure(figsize=(10, 6))

plt.errorbar(airmass[~mascara], diff_mags[~mascara], yerr=mag_err[~mascara], 
             fmt='o', color='lightgray', label='Descartados', alpha=0.5)

plt.errorbar(airmass_f, diff_f, yerr=err_f, 
             fmt='o', color='darkblue', ecolor='gray', capsize=3, label='Usados para el ajuste')

x_plot = np.linspace(1.0, 2.2, 100)
y_plot = modelo_extincion(x_plot, k_val, zp_val)
plt.plot(x_plot, y_plot, 'r--', label=f'Ajuste: k = {k_val:.3f} ± {err_k:.3f}')

plt.gca().invert_yaxis() # Invertir eje Y (Magnitudes)
plt.gca().yaxis.set_major_locator(MultipleLocator(0.2)) # Marcas cada 0.2 mag
plt.xlabel('Masa de Aire (X)')
plt.ylabel('$m_{std} - m_{inst}$')
plt.title('Solución Fotométrica (Filtro V)')
plt.legend()
plt.grid(alpha=0.2)

plt.show()

print(f"Resultado: k = {k_val:.4f} ± {err_k:.4f} mag/airmass")
print(f"Término independiente (ZP): {zp_val:.4f} ± {err_zp:.4f} mag")

Resultado: k = 0.2263 ± 0.0227 mag/airmass
Término independiente (ZP): 4.4116 ± 0.0344 mag


In [303]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from matplotlib.ticker import MultipleLocator

datos = np.loadtxt('rrr.txt')
diff_mags = datos[:, 0]
mag_err = np.abs(datos[:, 1])
airmass = datos[:, 2]

mascara = (diff_mags > 4)

airmass_f = airmass[mascara]
diff_f = diff_mags[mascara]
err_f = mag_err[mascara]

def modelo_extincion(x, k, ZP):
    return ZP - k * x

popt, pcov = curve_fit(modelo_extincion, airmass_f, diff_f, sigma=err_f)
k_val, zp_val = popt
err_k = np.sqrt(pcov[0,0])
err_zp = np.sqrt(pcov[1,1])

plt.figure(figsize=(10, 6))

plt.errorbar(airmass[~mascara], diff_mags[~mascara], yerr=mag_err[~mascara], 
             fmt='o', color='lightgray', label='Descartados', alpha=0.5)

plt.errorbar(airmass_f, diff_f, yerr=err_f, 
             fmt='o', color='darkblue', ecolor='gray', capsize=3, label='Usados para el ajuste')

x_plot = np.linspace(1.0, 2.2, 100)
y_plot = modelo_extincion(x_plot, k_val, zp_val)
plt.plot(x_plot, y_plot, 'r--', label=f'Ajuste: k = {k_val:.3f} ± {err_k:.3f}')

plt.gca().invert_yaxis() # Invertir eje Y (Magnitudes)
plt.gca().yaxis.set_major_locator(MultipleLocator(0.2)) # Marcas cada 0.2 mag
plt.xlabel('Masa de Aire (X)')
plt.ylabel('$m_{std} - m_{inst}$')
plt.title('Solución Fotométrica (Filtro R)')
plt.legend()
plt.grid(alpha=0.2)

plt.show()

print(f"Resultado: k = {k_val:.4f} ± {err_k:.4f} mag/airmass")
print(f"Término independiente (ZP): {zp_val:.4f} ± {err_zp:.4f} mag")

Resultado: k = 0.1203 ± 0.0303 mag/airmass
Término independiente (ZP): 4.2782 ± 0.0421 mag


In [304]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from matplotlib.ticker import MultipleLocator

datos = np.loadtxt('iii.txt')
diff_mags = datos[:, 0]
mag_err = np.abs(datos[:, 1])
airmass = datos[:, 2]

mascara = (diff_mags > 3.6)

airmass_f = airmass[mascara]
diff_f = diff_mags[mascara]
err_f = mag_err[mascara]

def modelo_extincion(x, k, ZP):
    return ZP - k * x

popt, pcov = curve_fit(modelo_extincion, airmass_f, diff_f, sigma=err_f)
k_val, zp_val = popt
err_k = np.sqrt(pcov[0,0])
err_zp = np.sqrt(pcov[1,1])

plt.figure(figsize=(10, 6))

plt.errorbar(airmass[~mascara], diff_mags[~mascara], yerr=mag_err[~mascara], 
             fmt='o', color='lightgray', label='Descartados', alpha=0.5)

plt.errorbar(airmass_f, diff_f, yerr=err_f, 
             fmt='o', color='darkblue', ecolor='gray', capsize=3, label='Usados para el ajuste')

x_plot = np.linspace(1.0, 2.2, 100)
y_plot = modelo_extincion(x_plot, k_val, zp_val)
plt.plot(x_plot, y_plot, 'r--', label=f'Ajuste: k = {k_val:.3f} ± {err_k:.3f}')

plt.gca().invert_yaxis() # Invertir eje Y (Magnitudes)
plt.gca().yaxis.set_major_locator(MultipleLocator(0.2)) # Marcas cada 0.2 mag
plt.xlabel('Masa de Aire (X)')
plt.ylabel('$m_{std} - m_{inst}$')
plt.title('Solución Fotométrica (Filtro I)')
plt.legend()
plt.grid(alpha=0.2)

plt.show()

print(f"Resultado: k = {k_val:.4f} ± {err_k:.4f} mag/airmass")
print(f"Término independiente (ZP): {zp_val:.4f} ± {err_zp:.4f} mag")

Resultado: k = 0.0820 ± 0.0380 mag/airmass
Término independiente (ZP): 3.8584 ± 0.0528 mag


**4.- Elije dos objetos de interés entre las imágenes que hay disponibles y 
calcula su magnitud instrumental y de al menos dos estrellas de
comparación. Hazlo para cada filtro y resume los resultados en una tabla**

In [289]:
#0051 y 0133 B

In [315]:
#visualizar la imagen
ima3 = 'DATA_06MAST/DATA_0071_mediana_bias_mediana_mediana_flat.fits'
imagen_sci3 = fits.getdata(ima3)
#llamado a la función:
visualizar_flat(imagen_sci3)

In [323]:
#cambiar la imagen y las coordenadas:
ima = 'DATA_06MAST/DATA_0071_mediana_bias_mediana_mediana_flat.fits'
archivo_coor = 'coor_0071_I.dat'

#leer la imagen:
imagen_sci = fits.getdata(ima)
header = fits.getheader(ima)
#gain = header['GAIN']
#r_noise = header['RDNOISE']
#t_exp = header['EXPTIME']
#air_mass = header['AIRMASS']

#leer las coordenadas
posiciones = np.loadtxt(archivo_coor)
posiciones

#recentrar:
coor_x, coor_y = centroid_sources(imagen_sci, posiciones[:,0], posiciones[:,1],
                                 box_size = 5, centroid_func = centroid_com)

#organizar las coordenadas en pares:
posiciones_centradas = np.transpose((coor_x, coor_y))

#preparar la apertura:
apertura = [CircularAperture(posiciones_centradas, r = 12)]

#anillo:
anillo = [CircularAnnulus(posiciones_centradas,r_in=20, r_out=25)]

#apertura total:
full_ap = apertura + anillo

#cálculo de desviación estándar para cada píxel:
error_map = calc_total_error(imagen_sci, bkg_error = header['RDNOISE'], effective_gain = header['GAIN'])

#fotometría de apertura:
fa = aperture_photometry(imagen_sci, full_ap, error = error_map)

#cielo/px:
cielo = fa['aperture_sum_1']/anillo[0].area

#cálculo de magnitud instrumental:
fa['mag_inst'] = -2.5*np.log10((fa['aperture_sum_0']-cielo*apertura[0].area)/header['EXPTIME'])+20

#error de la magnitud instrumental:
fa['mag_inst_err'] = 1.0857*np.sqrt(fa['aperture_sum_err_0']**2
                                    +((fa['aperture_sum_1']/anillo[0].area)*apertura[0].area)**2)/fa['aperture_sum_0']

#columna de masa de aire
fa['Masa_aire'] = header['AIRMASS']

#exportar la tabla:
nombre_tabla = f"mag_instr_{archivo_coor}.csv"
fa.write(nombre_tabla, format='ascii.csv', delimiter=';', overwrite=True)